# Jura Network — Interactive Condition Score Map
## Route Roulette · Hack4Rail 2026

Renders an interactive Folium map of the SBB Jura sub-network sections colour-coded by WBW-weighted condition score (green = best, red = worst).

**Prerequisites:** run `route_roulette_analysis.ipynb` first to generate `sec_summary.csv`.

## 1 · Import Required Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import re
import pandas as pd
import folium
from branca.colormap import LinearColormap

print("Libraries loaded ✓")

Libraries loaded ✓


## 2 · Load Section Summary Data

Loads `data/version_limited_ressources/C200/Jura_Entwicklung_Netz.xlsx` (sheet `default_1`) and aggregates the 2025 snapshot to one row per section.

| Column | Source | Meaning |
|---|---|---|
| `section_label` | `Linie Level 3` (cleaned) | Human-readable section name |
| `substanz_score` | `Substanz` (mean) | Mean condition score per section |
| `er_need_MCHF` | `Erneuerung` (sum) | Renewal cost demand (MCHF) |
| `unterhalt_MCHF` | `Unterhalt` (sum) | Maintenance cost (MCHF) |
| `strecken_cat` | `Streckenkategorie Level 1` | Route category |

In [2]:
DATA_FILE  = Path("data/version_limited_ressources/C200/Jura_Entwicklung_Netz.xlsx")
CACHE_FILE = DATA_FILE.with_suffix(".parquet")
SEC = "Linie Level 3"

# Read from Parquet cache if available (much faster than xlsx via openpyxl).
# Delete the .parquet file to force a re-read when the source Excel changes.
if CACHE_FILE.exists():
    df = pd.read_parquet(CACHE_FILE)
    print(f"Loaded from cache: {CACHE_FILE.name}")
else:
    print(f"Reading {DATA_FILE.name} (first time — will be cached as parquet)…")
    df = pd.read_excel(DATA_FILE, sheet_name="default_1", engine="openpyxl")
    df.to_parquet(CACHE_FILE, index=False)
    print(f"Cached → {CACHE_FILE.name}")

# Auto-detect availability and safety columns (names may vary by file version)
AVAIL_COL  = next((c for c in df.columns if "verfüg" in c.lower() or "avail"  in c.lower()), None)
SAFETY_COL = next((c for c in df.columns if "sicher" in c.lower() or "safety" in c.lower()), None)
print(f"Columns: {df.columns.tolist()}")
print(f"Availability col: {AVAIL_COL!r}  |  Safety col: {SAFETY_COL!r}")

# Use the 2025 snapshot (current inventory year)
df25 = df[df["Betrachtungsjahr"] == 2025].copy()

# Base aggregation — extend with availability/safety if present
agg_spec = {
    "substanz_score": ("Substanz", "mean"),
    "er_need_MCHF":   ("Erneuerung", lambda x: x.sum() / 1e6),
    "unterhalt_MCHF": ("Unterhalt",  lambda x: x.sum() / 1e6),
    "strecken_cat":   ("Streckenkategorie Level 1", "first"),
}
if AVAIL_COL:
    agg_spec["availability_score"] = (AVAIL_COL, "mean")
if SAFETY_COL:
    agg_spec["safety_score"] = (SAFETY_COL, "mean")

sec_summary = df25.groupby(SEC).agg(**agg_spec).reset_index()

sec_summary["section_label"] = (
    sec_summary[SEC]
    .str.replace(r"^\d+\s*-\s*", "", regex=True)
    .str.replace(r"[()]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print(f"\n{len(sec_summary)} sections · {df['Betrachtungsjahr'].nunique()} years in data")
display(sec_summary.head())

Loaded from cache: Jura_Entwicklung_Netz.parquet
Columns: ['Streckenkategorie Level 1', 'Linie Level 2', 'Linie Level 3', 'Betrachtungsjahr', 'Kostenstellen', 'Erneuerung', 'Unterhalt', 'Substanz']
Availability col: None  |  Safety col: None

27 sections · 26 years in data


,Linie Level 3,substanz_score,er_need_MCHF,unterhalt_MCHF,strecken_cat,section_label
0,200100 - (Renens) - (Bussigny),1.676651,39.141879,0.430956,Hauptstrecke,Renens - Bussigny
1,200150 - Bussigny - Daillens [bif],2.884510,1.761964,8.045645,Hauptstrecke,Bussigny - Daillens [bif]
2,200200 - (Daillens) - (Le Day),2.742018,1.694285,4.754748,Regionalstrecke,Daillens - Le Day
3,200250 - Le Day - (Vallorbe),3.082519,8.759388,2.660219,Regionalstrecke,Le Day - Vallorbe
4,200300 - Vallorbe (communauté),2.858887,14.397320,0.651017,Regionalstrecke,Vallorbe communauté


In [3]:
sec_summary["section_label"].unique()

<ArrowStringArray>
[                'Renens - Bussigny',         'Bussigny - Daillens [bif]',
                 'Daillens - Le Day',                 'Le Day - Vallorbe',
               'Vallorbe communauté',                  'Le Day - Le Pont',
     'Vallorbe frontière - Vallorbe',               'Auvernier - Travers',
 'Travers - Les Verrières-frontière',     'Neuchâtel - La Chaux-de-Fonds',
                 'La Chaux-de-Fonds',      'La Chaux-de-Fonds - Le Locle',
     'Le Locle - Le Locle frontière',    'Biel - Sonceboz-Sombeval Ouest',
      'Sonceboz - La Chaux-de-Fonds',                'Sonceboz - Moutier',
                'Moutier - Choindez',               'Choindez - Delémont',
          'Delémont - Soyhières Est',            'Soyhières Est - Laufen',
                    'Laufen - Aesch',              'Delémont - Glovelier',
            'Glovelier - Porrentruy',         'Porrentruy - Courtemaîche',
           'Courtemaîche - Boncourt',        'Boncourt - Delle Frontière',
      

## 3 · Station Coordinate Lookup

In [4]:
# WGS84 (lat, lon) for stations on the Jura sub-network (lines 200–230)
STATION_COORDS = {
    "lausanne":             (46.5197,  6.6323),
    "renens":               (46.5331,  6.5858),
    "morges":               (46.5098,  6.4985),
    "yverdon":              (46.7784,  6.6408),
    "yverdon-les-bains":    (46.7784,  6.6408),
    "grandson":             (46.8082,  6.6503),
    "concise":              (46.8538,  6.7284),
    "neuchatel":            (47.0002,  6.9327),
    "neuchâtel":            (47.0002,  6.9327),
    "le landeron":          (47.0558,  7.0670),
    "cressier":             (47.0494,  7.0469),
    "biel":                 (47.1366,  7.2464),
    "bienne":               (47.1366,  7.2464),
    "biel/bienne":          (47.1366,  7.2464),
    "lyss":                 (47.0727,  7.3055),
    "kerzers":              (46.9891,  7.1956),
    "murten":               (46.9247,  7.1200),
    "morat":                (46.9247,  7.1200),
    "payerne":              (46.8207,  6.9383),
    "avenches":             (46.8781,  7.0433),
    "lengnau":              (47.1836,  7.3858),
    "bettlach":             (47.2047,  7.4089),
    "grenchen":             (47.1922,  7.3960),
    "grenchen nord":        (47.1977,  7.3938),
    "grenchen sud":         (47.1877,  7.3985),
    "solothurn":            (47.2088,  7.5367),
    "soleure":              (47.2088,  7.5367),
    "olten":                (47.3524,  7.9074),
    "aarau":                (47.3912,  8.0441),
    "basel":                (47.5596,  7.5886),
    "bale":                 (47.5596,  7.5886),
    "bâle":                 (47.5596,  7.5886),
    "laufen":               (47.4261,  7.4975),
    "aesch":                (47.4803,  7.5940),
    "dornach":              (47.4779,  7.6140),
    "liestal":              (47.4839,  7.7315),
    "pratteln":             (47.5220,  7.6999),
    "delemont":             (47.3666,  7.3446),
    "delémont":             (47.3666,  7.3446),
    "glovelier":            (47.3369,  7.2106),
    "porrentruy":           (47.4160,  7.0755),
    "boncourt":             (47.4899,  7.0050),
    "delle":                (47.5073,  6.9935),
    "courrendlin":          (47.3395,  7.3668),
    "courroux":             (47.3587,  7.3545),
    "bassecourt":           (47.3292,  7.2449),
    "undervelier":          (47.2989,  7.2056),
    "moutier":              (47.2772,  7.3695),
    "munster":              (47.2666,  7.3796),
    "münster":              (47.2666,  7.3796),
    "court":                (47.2530,  7.3564),
    "reconvilier":          (47.2381,  7.3200),
    "sonceboz":             (47.2020,  7.2860),
    "sonceboz-sombeval":    (47.2020,  7.2860),
    "tavannes":             (47.2200,  7.2070),
    "la chaux-de-fonds":    (47.0956,  6.8275),
    "chaux-de-fonds":       (47.0956,  6.8275),
    "la chaux de fonds":    (47.0956,  6.8275),
    "le locle":             (47.0563,  6.7519),
    "les ponts":            (46.9881,  6.6280),
    "saint-imier":          (47.1528,  6.9990),
    "st-imier":             (47.1528,  6.9990),
    "tramelan":             (47.2107,  7.1008),
    "saignelegier":         (47.2566,  7.0024),
    "saignelégier":         (47.2566,  7.0024),
    "le noirmont":          (47.2269,  6.9541),
    "noirmont":             (47.2269,  6.9541),
    "vallorbe":             (46.7136,  6.3717),
    "pontarlier":           (46.9052,  6.3549),
    "bern":                 (46.9481,  7.4474),
    "berne":                (46.9481,  7.4474),
    "zollikofen":           (47.0172,  7.4594),
    "münchenbuchsee":       (47.0280,  7.4280),
    "courtemaîche":         (47.45953, 7.04741),
    "courtemaiche":         (47.45953, 7.04741),
    }

def _norm(s: str) -> str:
    """Lowercase + strip accents for fuzzy key matching."""
    return (s.lower()
             .replace("ü", "u").replace("ö", "o").replace("ä", "a")
             .replace("é", "e").replace("è", "e").replace("ê", "e")
             .replace("â", "a").replace("î", "i").strip())

print(f"Coordinate lookup: {len(STATION_COORDS)} stations")

Coordinate lookup: 75 stations


## 4 · Helper Functions

In [5]:
def section_to_coords(label: str) -> list[tuple[float, float]]:
    """
    Split a section label on delimiters (- – /) and resolve each token
    to WGS84 coordinates via STATION_COORDS.
    Returns a list of (lat, lon) tuples — at least 2 for a PolyLine.
    """
    parts = re.split(r"\s*[-–/]\s*", label)
    coords = []
    for part in parts:
        key = _norm(part)
        if key in STATION_COORDS:
            coords.append(STATION_COORDS[key])
            continue
        # fuzzy substring match
        match = next((c for k, c in STATION_COORDS.items() if k in key or key in k), None)
        if match:
            coords.append(match)
    return coords

## 5 · Build and Render Interactive Map

Sections drawn as straight lines between endpoint stations, colour-coded **green → yellow → red** for best → worst condition score. Hover over a line or marker to see full KPIs. The map is also saved as `section_map.html`.

In [6]:
# ── Fixed colour scale 1–5 (consistent across all maps) ──────────────────────
colormap = LinearColormap(
    colors=["#2ecc71", "#f1c40f", "#e74c3c", "#8B0000"],
    index=[1, 3, 4, 5],
    vmin=1, vmax=5,
    caption="Condition score  1 = best (green) · 4 = red · 5 = dark red",
)

# ── Build map ─────────────────────────────────────────────────────────────────
m = folium.Map(location=(47.1, 7.1), zoom_start=8, tiles="OpenStreetMap")
colormap.add_to(m)

matched, unmatched = [], []

for _, row in sec_summary.iterrows():
    label = row["section_label"]
    score = row["substanz_score"]
    if pd.isna(score):
        continue

    coords = section_to_coords(label)
    color  = colormap(score)

    tip_lines = [
        f"<b>{label}</b>",
        f"Condition score: <b>{score:.2f}</b>",
        f"Renewal (ER): {row['er_need_MCHF']:.2f} MCHF",
        f"Maintenance: {row['unterhalt_MCHF']:.2f} MCHF",
    ]
    if AVAIL_COL and "availability_score" in row.index and not pd.isna(row.get("availability_score")):
        tip_lines.append(f"Availability: {row['availability_score']:.2f}")
    if SAFETY_COL and "safety_score" in row.index and not pd.isna(row.get("safety_score")):
        tip_lines.append(f"Safety: {row['safety_score']:.2f}")
    tip_lines.append(f"Category: {row['strecken_cat']}")
    tip = "<br>".join(tip_lines)

    if len(coords) >= 2:
        folium.PolyLine(
            locations=coords, color=color, weight=7, opacity=0.9,
            tooltip=folium.Tooltip(tip, sticky=True),
        ).add_to(m)
        for c in [coords[0], coords[-1]]:
            folium.CircleMarker(
                location=c, radius=5, color=color,
                fill=True, fill_color=color, fill_opacity=1.0,
                tooltip=folium.Tooltip(tip, sticky=True),
            ).add_to(m)
        matched.append(label)
    elif len(coords) == 1:
        folium.CircleMarker(
            location=coords[0], radius=9, color=color,
            fill=True, fill_color=color, fill_opacity=0.9,
            tooltip=folium.Tooltip(tip, sticky=True),
        ).add_to(m)
        matched.append(f"{label} (point only)")
    else:
        unmatched.append(label)

print(f"Sections rendered : {len(matched)}/{len(sec_summary)}")
if unmatched:
    print("Unmatched (add to STATION_COORDS to fix):")
    for u in unmatched:
        print(f"  {u}")

m.save("section_map.html")
print("Map saved → section_map.html")
display(m)


Sections rendered : 22/27
Unmatched (add to STATION_COORDS to fix):
  Bussigny - Daillens [bif]
  Daillens - Le Day
  Le Day - Le Pont
  Auvernier - Travers
  Travers - Les Verrières-frontière
Map saved → section_map.html


## 6 · Animated Map — Substanzqualität Over Time

Sections are colour-coded **green → yellow → red** for best → worst condition score. Use the time slider (or ▶ play button) to step through years. Also saved as `section_map_animated.html`.

In [7]:
import json
from folium.plugins import TimestampedGeoJson
from branca.element import MacroElement
from jinja2 import Template

# ── Aggregate all years ───────────────────────────────────────────────────────
agg_spec_all = {
    "substanz_score": ("Substanz", "mean"),
    "er_need_MCHF":   ("Erneuerung", lambda x: x.sum() / 1e6),
    "unterhalt_MCHF": ("Unterhalt",  lambda x: x.sum() / 1e6),
    "strecken_cat":   ("Streckenkategorie Level 1", "first"),
}
if AVAIL_COL:
    agg_spec_all["availability_score"] = (AVAIL_COL, "mean")
if SAFETY_COL:
    agg_spec_all["safety_score"] = (SAFETY_COL, "mean")

agg_all = df.groupby([SEC, "Betrachtungsjahr"]).agg(**agg_spec_all).reset_index()
agg_all["section_label"] = (
    agg_all[SEC]
    .str.replace(r"^\d+\s*-\s*", "", regex=True)
    .str.replace(r"[()]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

years = sorted(agg_all["Betrachtungsjahr"].dropna().unique().astype(int).tolist())
print(f"Years in data: {years[0]}–{years[-1]} ({len(years)} steps)")

# ── Fixed colour scale 1–5 ────────────────────────────────────────────────────
cmap_anim = LinearColormap(
    colors=["#2ecc71", "#f1c40f", "#e74c3c", "#8B0000"],
    index=[1, 3, 4, 5],
    vmin=1, vmax=5,
    caption="Quality score  (1: best · 4: red · 5: dark red)",
)

# ── Per-year stats ────────────────────────────────────────────────────────────
year_stats = {
    int(yr): {
        "min": round(float(g["substanz_score"].min()), 2),
        "max": round(float(g["substanz_score"].max()), 2),
    }
    for yr, g in agg_all.groupby("Betrachtungsjahr")
}

# ── Build GeoJSON features ────────────────────────────────────────────────────
features = []
for _, row in agg_all.iterrows():
    label = row["section_label"]
    score = row["substanz_score"]
    year  = int(row["Betrachtungsjahr"])
    if pd.isna(score):
        continue
    coords = section_to_coords(label)
    if len(coords) < 2:
        continue

    tip_lines = [
        f"<b>{label}</b>",
        f"Year: <b>{year}</b>",
        f"Condition: <b>{score:.2f}</b>",
        f"Renewal: {row['er_need_MCHF']:.2f} MCHF",
        f"Maintenance: {row['unterhalt_MCHF']:.2f} MCHF",
    ]
    if AVAIL_COL and "availability_score" in row.index and not pd.isna(row.get("availability_score")):
        tip_lines.append(f"Availability: {row['availability_score']:.2f}")
    if SAFETY_COL and "safety_score" in row.index and not pd.isna(row.get("safety_score")):
        tip_lines.append(f"Safety: {row['safety_score']:.2f}")
    tip_lines.append(f"Category: {row['strecken_cat']}")

    features.append({
        "type": "Feature",
        "geometry": {"type": "LineString", "coordinates": [[c[1], c[0]] for c in coords]},
        "properties": {
            "times":  [f"{year}-01-01", f"{year + 1}-01-01"],
            "style":  {"color": cmap_anim(score), "weight": 7, "opacity": 0.9},
            "popup":  "<br>".join(tip_lines),
        },
    })

# ── Assemble map ──────────────────────────────────────────────────────────────
m_anim = folium.Map(location=(47.1, 7.1), zoom_start=8, tiles="CartoDB positron")
cmap_anim.add_to(m_anim)

# Larger legend text (branca renders SVG text; !important overrides inline sizes)
m_anim.get_root().header.add_child(folium.Element(
    "<style>"
    ".legend text { font-size: 14px !important; font-weight: 600 !important; }"
    ".legend .tick text { font-size: 12px !important; }"
    "</style>"
))

TimestampedGeoJson(
    {"type": "FeatureCollection", "features": features},
    period="P1Y",
    add_last_point=False,
    auto_play=False,
    loop=False,
    loop_button=True,
    date_options="YYYY",
    time_slider_drag_update=True,
).add_to(m_anim)

# ── Year box via MacroElement (runs after map + TimestampedGeoJson are ready) ─
class YearOverlay(MacroElement):
    def __init__(self, year_stats, first_yr):
        super().__init__()
        self._name = "YearOverlay"
        self.year_stats_json = json.dumps(year_stats)
        self.first_yr = str(first_yr)
        first_stats = year_stats.get(first_yr, {"min": 1.0, "max": 5.0})
        self.first_range = f"Best: {first_stats['min']:.2f}&nbsp;&nbsp;Worst: {first_stats['max']:.2f}"
        self._template = Template("""
            {% macro script(this, kwargs) %}
            (function() {
                var S      = {{ this.year_stats_json }};
                var mapObj = {{ this._parent.get_name() }};

                var YrCtrl = L.control({position: "topright"});
                YrCtrl.onAdd = function() {
                    var d = L.DomUtil.create("div", "yr-overlay-box");
                    d.style.cssText = "background:rgba(255,255,255,0.93);padding:10px 16px;"
                                    + "border-radius:10px;border:2px solid #bbb;"
                                    + "font-family:Arial,sans-serif;text-align:center;"
                                    + "box-shadow:2px 2px 8px rgba(0,0,0,0.2);"
                                    + "min-width:140px;margin-top:70px;";
                    d.innerHTML = "<div id='yr-num' style='font-size:2.6em;font-weight:bold;"
                                + "color:#222;line-height:1.1;'>{{ this.first_yr }}</div>"
                                + "<div id='yr-range' style='font-size:0.78em;color:#555;"
                                + "margin-top:4px;'>{{ this.first_range }}</div>";
                    L.DomEvent.disableClickPropagation(d);
                    return d;
                };
                YrCtrl.addTo(mapObj);

                function setBox(yr) {
                    yr = parseInt(yr, 10);
                    if (!yr || !S[yr]) return;
                    var n = document.getElementById("yr-num");
                    var r = document.getElementById("yr-range");
                    if (n) n.textContent = yr;
                    if (r) r.innerHTML = "Best:&nbsp;" + S[yr].min.toFixed(2)
                                       + "&nbsp;&nbsp;Worst:&nbsp;" + S[yr].max.toFixed(2);
                }

                var td = mapObj.timeDimension;
                if (td && typeof td.on === "function") {
                    var t0 = td.getCurrentTime();
                    if (t0) setBox(new Date(t0).getUTCFullYear());
                    td.on("timeload", function(e) {
                        setBox(new Date(e.time || td.getCurrentTime()).getUTCFullYear());
                    });
                }

                setInterval(function() {
                    var el = document.querySelector(".leaflet-control-timecontrol-date");
                    if (!el) return;
                    var yr = parseInt(el.textContent.trim(), 10);
                    if (!isNaN(yr) && yr > 2000) setBox(yr);
                }, 200);
            })();
            {% endmacro %}
        """)

m_anim.add_child(YearOverlay(year_stats, years[0]))

print(f"Features built: {len(features)}")
m_anim.save("section_map_animated.html")
print("Saved → section_map_animated.html")
display(m_anim)


Years in data: 2025–2050 (26 steps)
Features built: 364
Saved → section_map_animated.html
